# Hyperparameter tuning XGBOOST (readmission as a reference)

## 0. Package loading and installation

In [1]:
# Commented out IPython magic to ensure Python compatibility.
# For Jupyter/Colab notebooks
%reset -f
import gc
gc.collect()

import numpy as np
import pandas as pd
import time

#conda install -c conda-forge \
#    numpy \
#    scipy \
#    pandas \
#    pyarrow \
#    scikit-survival \
#    spyder \
#    lifelines

# conda install -c conda-forge fastparquet
# conda install -c conda-forge xgboost
# conda install -c conda-forge pytorch cpuonly
# conda install -c pytorch pytorch cpuonly
# conda install -c conda-forge matplotlib
# conda install -c conda-forge seaborn
# conda install spyder-notebook -c spyder-ide
# conda install notebook nbformat nbconvert
# conda install -c conda-forge xlsxwriter
# conda install -c conda-forge shap

# import subprocess, sys

# subprocess.check_call([
#     sys.executable,
#     "-m",
#     "pip",
#     "install",
#     "matplotlib"
# ])

# subprocess.check_call([
#     sys.executable,
#     "-m",
#     "pip",
#     "install",
#     "seaborn"
# ])

print("numpy:", np.__version__)


from sksurv.metrics import (
    concordance_index_ipcw,
    brier_score,
    integrated_brier_score
)
from sksurv.util import Surv

#Dput
def dput_df(df, digits=6):
    data = {
        "columns": list(df.columns),
        "data": [
            [round(x, digits) if isinstance(x, (float, np.floating)) else x
             for x in row]
            for row in df.to_numpy()
        ]
    }
    print(data)


#Glimpse function
def glimpse(df, max_width=80):
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    for col in df.columns:
        dtype = df[col].dtype
        preview = df[col].astype(str).head(5).tolist()
        preview_str = ", ".join(preview)
        if len(preview_str) > max_width:
            preview_str = preview_str[:max_width] + "..."
        print(f"{col:<30} {str(dtype):<15} {preview_str}")
#Tabyl function
def tabyl(series):
    counts = series.value_counts(dropna=False)
    props = series.value_counts(normalize=True, dropna=False)
    return pd.DataFrame({"value": counts.index,
                         "n": counts.values,
                         "percent": props.values})
#clean_names
import re

def clean_names(df):
    """
    Mimic janitor::clean_names for pandas DataFrames.
    - Lowercase
    - Replace spaces and special chars with underscores
    - Remove non-alphanumeric/underscore
    """
    new_cols = []
    for col in df.columns:
        # lowercase
        col = col.lower()
        # replace spaces and special chars with underscore
        col = re.sub(r"[^\w]+", "_", col)
        # strip leading/trailing underscores
        col = col.strip("_")
        new_cols.append(col)
    df.columns = new_cols
    return df

numpy: 2.0.1


## Load data

In [2]:

from pathlib import Path

BASE_DIR = Path(
    r"G:\My Drive\Alvacast\SISTRAT 2023\data\20241015_out\pred1"
)


import pickle

with open(BASE_DIR / "imputations_list_jan26.pkl", "rb") as f:
    imputations_list_jan26 = pickle.load(f)


imputation_nodum_1 = pd.read_parquet(
    BASE_DIR / "imputation_nondum_1.parquet",
    engine="fastparquet"
)

X_reduced_imp0 = pd.read_parquet(
    BASE_DIR / "X_reduced_imp0.parquet",
    engine="fastparquet"
)

imputation_1 = pd.read_parquet(
    BASE_DIR / "imputation_1.parquet",
    engine="fastparquet"
)

In [3]:
from IPython.display import display, HTML
import io
import sys

def fold_output(title, func):
    buffer = io.StringIO()
    sys.stdout = buffer
    func()
    sys.stdout = sys.__stdout__
    
    html = f"""
    <details>
      <summary>{title}</summary>
      <pre>{buffer.getvalue()}</pre>
    </details>
    """
    display(HTML(html))


fold_output(
    "Show imputation_nodum_1 structure",
    lambda: imputation_nodum_1.info()
)

fold_output(
    "Show imputation_1 structure",
    lambda: imputation_1.info()
)

fold_output(
    "Show X_reduced_imp0 structure",
    lambda: X_reduced_imp0.info()
)

In [4]:
if isinstance(imputations_list_jan26, list) and len(imputations_list_jan26) > 0:
    print("First element type:", type(imputations_list_jan26[0]))
    if isinstance(imputations_list_jan26[0], dict):
        print("First element keys:", imputations_list_jan26[0].keys())
    elif isinstance(imputations_list_jan26[0], (pd.DataFrame, np.ndarray)):
        print("First element shape:", imputations_list_jan26[0].shape)


This code block:

1.  **Imports the `pickle` library**: This library implements binary protocols for serializing and de-serializing a Python object structure.
2.  **Specifies the `file_path`**: It points to the `.pkl` file you selected.
3.  **Opens the file in binary read mode (`'rb'`)**: This is necessary for loading pickle files.
4.  **Loads the object**: `pickle.load(f)` reads the serialized object from the file and reconstructs it in memory.
5.  **Prints confirmation and basic information**: It verifies that the file was loaded and shows the type of the loaded object, and some details about the first element if it's a list containing common data structures.

#### Compare databases (transformed and original)

Inspect and compare the column names of two datasets: the first imputation from imputations_list_jan26 (which likely contains dummy variables) and imputation_nodum_1 (which, as its name suggests, probably doesn't have dummy variables).


In [5]:
# Inspect columns of the first imputation
cols_first_imp = imputations_list_jan26[0].columns.tolist()
print("First imputation columns:", cols_first_imp[:10], "... total:", len(cols_first_imp))

# Inspect columns of imputation_no_dum
cols_nodum = imputation_nodum_1.columns.tolist()
print("No-dum columns:", cols_nodum[:10], "... total:", len(cols_nodum))

# Compare overlap
common_cols = set(cols_first_imp).intersection(cols_nodum)
missing_in_imp = [c for c in cols_nodum if c not in cols_first_imp]
missing_in_nodum = [c for c in cols_first_imp if c not in cols_nodum]

print("Common columns:", len(common_cols))
print("Missing in imputations_list_jan26:", missing_in_imp)

In [6]:
# Inspect columns of the first imputation
cols_first_imp_raw = imputation_1.columns.tolist()
print("First imputation columns:", cols_first_imp_raw[:10], "... total:", len(cols_first_imp_raw))

# Compare overlap
common_cols_raw = set(cols_first_imp_raw).intersection(cols_nodum)
missing_in_imp_raw = [c for c in cols_nodum if c not in cols_first_imp_raw]

print("Common columns:", len(common_cols_raw))
print("Missing in imputations_list_jan26:", missing_in_imp_raw)
print(common_cols_raw)

In [7]:
import pandas as pd

# Example: choose a combination of variables that uniquely identify rows
key_vars = ["adm_age_rec3", "porc_pobr", "dit_m"]

# Take one imputation (first element of the list) and merge with the no-dum dataset
df_imp = imputations_list_jan26[0]
df_nodum = imputation_nodum_1

merged_check = pd.merge(
    df_imp[key_vars],
    df_nodum[key_vars],
    on=key_vars,
    how="inner"
)

print(f"Merged rows: {merged_check.shape[0]}")
print("Preview of merged check:")
print(merged_check.head())

#drop merge
del merged_check

In [8]:
import pandas as pd

# Example: choose a combination of variables that uniquely identify rows
key_vars_raw = ['dit_m',
            'readmit_time_from_adm_m',
            'death_time_from_adm_m',
            'adm_age_rec3']
# Take one imputation (first element of the list) and merge with the no-dum dataset
df_raw = imputation_1

merged_check_raw = pd.merge(
    df_imp[key_vars],
    df_raw[key_vars],
    on=key_vars,
    how="inner"
)

print(f"Merged rows: {merged_check_raw.shape[0]}")
print("Preview of merged check:")
print(merged_check_raw.head())
print(f"{(merged_check_raw.shape[0] / imputation_1.shape[0] * 100):.2f}%")
#drop merge
del merged_check_raw

### Create bins for followup (landmarks)

This code prepares your data for survival analysis. It extracts the time until an event (like readmission or death) and whether that event actually happened for each patient from the df_nodum dataset. Then, it automatically creates a set of important time points, called an 'evaluation grid', which are specific moments to assess the model's performance on both readmission and death outcomes.


In [9]:
import numpy as np

# Required columns for survival outcomes
required = ["readmit_time_from_disch_m", "readmit_event",
            "death_time_from_disch_m", "death_event"]

# Check that df_raw has all required columns
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise KeyError(f"df_nodum is missing columns: {missing}")

# Create time/event arrays directly from df_raw
time_readm = df_raw["readmit_time_from_disch_m"].to_numpy()
event_readm = (df_raw["readmit_event"].to_numpy() == 1)

time_death = df_raw["death_time_from_disch_m"].to_numpy()
event_death = (df_nodum["death_event"].to_numpy() == 1)

print("Arrays created for df_raw:")
print("Readmission times:", time_readm[:5])
print("Readmission events:", event_readm[:5])
print("Death times:", time_death[:5])
print("Death events:", event_death[:5])

# Build evaluation grids (quantiles of event times)
event_times_readm = time_readm[event_readm]
event_times_death = time_death[event_death]

if len(event_times_readm) < 5 or len(event_times_death) < 5:
    raise ValueError("Too few events in df_raw to build reliable time grids.")

times_eval_readm = np.unique(np.quantile(event_times_readm, np.linspace(0.05, 0.95, 50)))
times_eval_death = np.unique(np.quantile(event_times_death, np.linspace(0.05, 0.95, 50)))

print("Eval times (readmission):", times_eval_readm[:5], "...", times_eval_readm[-5:])
print("Eval times (death):", times_eval_death[:5], "...", times_eval_death[-5:])


 ## Prepare data


First, we eliminated inmortal time bias (dead patients look like without readmission).

This correction is essentially the Cause-Specific Hazard preparation. It is the correct way to handle Aim 3 unless you switch to a Fine-Gray model (which treats death as a specific type of event 2, rather than censoring 0). For RSF/Coxnet, censoring 0 is the correct approach.

In [10]:
import numpy as np

# Step 1. Extract survival outcomes directly from df_raw
time_readm = df_raw["readmit_time_from_disch_m"].to_numpy()
event_readm = (df_raw["readmit_event"].to_numpy() == 1)

time_death = df_raw["death_time_from_disch_m"].to_numpy()
event_death = (df_raw["death_event"].to_numpy() == 1)

# Step 2. Build structured arrays (Surv objects)
y_surv_readm = np.empty(len(time_readm), dtype=[("event", "?"), ("time", "<f8")])
y_surv_readm["event"] = event_readm
y_surv_readm["time"] = time_readm

y_surv_death = np.empty(len(time_death), dtype=[("event", "?"), ("time", "<f8")])
y_surv_death["event"] = event_death
y_surv_death["time"] = time_death

# Step 3. Replicate across imputations
n_imputations = len(imputations_list_jan26)
y_surv_readm_list = [y_surv_readm for _ in range(n_imputations)]
y_surv_death_list = [y_surv_death for _ in range(n_imputations)]

import numpy as np

def correct_competing_risks(X_list, y_readm_list, y_death_list):
    """
    Adjust survival outcomes for competing risks (death vs. readmission).

    Parameters
    ----------
    X_list : list of pd.DataFrame
        Imputed predictor datasets (same rows across imputations).
    y_readm_list : list of structured arrays
        Surv(event, time) arrays for readmission.
    y_death_list : list of structured arrays
        Surv(event, time) arrays for death.

    Returns
    -------
    y_readm_corrected_list : list of structured arrays
        Corrected readmission outcomes (death treated as censoring).
    """
    corrected = []
    for y_readm, y_death in zip(y_readm_list, y_death_list):
        y_corr = y_readm.copy()
        # If patient died before readmission → censor at death time
        for i in range(len(y_corr)):
            if y_death["event"][i] and y_death["time"][i] < y_corr["time"][i]:
                y_corr["event"][i] = False
                y_corr["time"][i] = y_death["time"][i]
        corrected.append(y_corr)
    return corrected


# Step 4. Apply correction
y_surv_readm_list_corrected = correct_competing_risks(
    imputations_list_jan26,
    y_surv_readm_list,
    y_surv_death_list
)

In [11]:
# Check type and length
type(y_surv_readm_list_corrected), len(y_surv_readm_list_corrected)

# Look at the first element
y_surv_readm_list_corrected[0][:5]   # first 5 rows

array([(False, 68.96774194), ( True,  7.        ), ( True, 13.25806452),
       ( True,  5.        ), ( True,  7.35483871)],
      dtype=[('event', '?'), ('time', '<f8')])

In [12]:

fold_output(
    "Show glimpse of imputations_list_jan26  (first imputation)",
    lambda: glimpse(imputations_list_jan26[0])
)

In [18]:
from IPython.display import display, HTML
import html

def nb_print(*args, sep=" "):
    msg = sep.join(str(a) for a in args)
    display(HTML(f"<pre style='margin:0'>{html.escape(msg)}</pre>"))


In [19]:
nb_print(y_surv_readm_list_corrected[0].shape, y_surv_readm_list_corrected[0].dtype)

# ML

### Advanced Survival Modeling: XGBoost & Stratified Evaluation

In this section, we transition to a Gradient Boosted Decision Tree (GBDT) framework using XGBoost. This approach serves as a robust, non-linear benchmark to validate findings from the neural network, specifically optimized for high-imbalance survival data (approx. 4% death rate).

#### Methodological Framework:
* **Cox-Objective Boosting:** We utilize the `survival:cox` objective, which optimizes the Cox partial log-likelihood within a boosting architecture. This allows the model to learn complex non-linear risk functions and interactions without assuming proportional hazards or requiring manual interaction terms.

* **Stratified 5-Fold Cross-Validation:** To ensure robustness across diverse treatment modalities, we implement `StratifiedKFold` based on Plan Type (Outpatient, Intensive, Residential). This guarantees that every validation fold maintains the same distribution of clinical settings as the full dataset, preventing the model from overfitting to the majority treatment type.

* **Robust Metrics (IPCW & IBS):** Instead of standard AUC, we optimize for **Uno's C-Index (Inverse Probability of Censoring Weighting)**. This metric is statistically consistent for censored data and prevents bias when evaluating long-term outcomes in unbalanced datasets. We additionally calculate the **Integrated Brier Score (IBS)** to assess the calibration of the predicted survival probabilities.

#### Hyperparameter Optimization:
Given the extreme class imbalance, we employ a **Stratified Randomized Search** over a dense parameter grid. This process tunes critical regularization parameters (`min_child_weight`, `gamma`, `reg_alpha`) to prevent overfitting to the majority class (survivors) while maximizing discrimination on the minority class (events).

#### Breslow Estimation:
To bridge the gap between XGBoost's raw risk scores (log-hazards) and interpretable probabilities needed for calibration metrics, we explicitly compute the **Breslow Estimator**. This reconstructs the baseline survival function S0(t), allowing us to project absolute survival probabilities S(t|x) for any patient at any time point.

In [21]:
#@title ⚡ XGBoost Readmission Robust Tuning (100 Iterations, CPU Only)
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, ParameterSampler
from sksurv.metrics import concordance_index_ipcw
import time
import gc
import os
from datetime import datetime
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

# Fallback in case nb_print is not defined globally
if 'nb_print' not in globals():
    def nb_print(*args, **kwargs):
        print(*args, **kwargs)

# Start Timer
total_start_time = time.time()

# --- CPU CONFIGURATION ---
# Calculate total cores minus 2 (ensuring at least 1 core is used)
N_CORES = max(1, os.cpu_count() - 2)
nb_print(f"⚙️ Parallel Execution Configured: Using {N_CORES} CPU cores.")

# --- 1. SETUP & DATA ---
nb_print("Preparing data for Robust XGBoost Tuning (Readmission)...")

try:
    if 'imputations_list_jan26' in locals():
        df_tune = imputations_list_jan26[0].copy()
        y_tune_struct = y_surv_readm_list_corrected[0]
    elif 'imputations_list' in locals():
        df_tune = imputations_list[0].copy()
        y_tune_struct = y_surv_readm_list_corrected[0]
    else:
        # Fallback
        df_tune = X_train.copy()
        y_tune_struct = y_surv_readm_list_corrected[0]

    nb_print(f"  Data Shape: {df_tune.shape}")
    nb_print(f"  Target: Readmission (Events: {y_tune_struct['event'].sum()})")

except Exception as e:
    raise ValueError(f"Data Error: {e}. Please run data loading steps first.")

# --- 2. STRATIFICATION HELPER ---
def get_stratification_labels(df):
    labels = np.zeros(len(df), dtype=int)
    if 'plan_type_corr_pg_pr' in df.columns: labels[df['plan_type_corr_pg_pr'] == 1] = 1
    if 'plan_type_corr_m_pr' in df.columns: labels[df['plan_type_corr_m_pr'] == 1] = 2
    if 'plan_type_corr_pg_pai' in df.columns: labels[df['plan_type_corr_pg_pai'] == 1] = 3
    if 'plan_type_corr_m_pai' in df.columns: labels[df['plan_type_corr_m_pai'] == 1] = 4
    return labels

strat_labels = get_stratification_labels(df_tune)
y_xgb_label = np.where(y_tune_struct['event'], y_tune_struct['time'], -y_tune_struct['time'])

# --- 3. EXHAUSTIVE SEARCH SPACE ---
param_grid = {
    'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    'max_depth': [3, 4, 5, 6, 8],
    'min_child_weight': [1, 5, 10, 20, 50],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.1, 1, 5, 10],
    'reg_lambda': [0.1, 1, 5, 10, 20],
    'gamma': [0, 0.1, 0.5, 1, 2]
}

N_ITER = 100
param_list = list(ParameterSampler(param_grid, n_iter=N_ITER, random_state=2125))

# --- 4. TUNING LOOP ---
nb_print(f"\n🚀 Starting Exhaustive Search ({N_ITER} combos)...")
nb_print(f"  Strategy: 5-Fold Stratified CV")
nb_print(f"  Metric: Uno's C-Index (IPCW)")

results = []

for i, params in enumerate(param_list):
    iter_start = time.time()

    # Fixed Parameters & Configuration (Strictly CPU)
    params['objective'] = 'survival:cox'
    params['eval_metric'] = 'cox-nloglik'
    params['tree_method'] = 'hist'
    params['seed'] = 2125            # Explicit seed for XGBoost reproducibility
    params['nthread'] = N_CORES    # Explicit CPU threading (Cores - 2)
    params['device'] = 'cpu'       # Hardcoded to CPU, removing GPU overrides
    params['verbosity'] = 0

    # 5-Fold matching your methodology
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2125)
    fold_scores = []

    for train_idx, val_idx in skf.split(df_tune, strat_labels):
        X_tr, X_va = df_tune.iloc[train_idx], df_tune.iloc[val_idx]
        y_tr_xgb, y_va_xgb = y_xgb_label[train_idx], y_xgb_label[val_idx]
        y_tr_struct, y_va_struct = y_tune_struct[train_idx], y_tune_struct[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr_xgb)
        dval = xgb.DMatrix(X_va, label=y_va_xgb)

        model = xgb.train(params, dtrain, num_boost_round=1500,
                          evals=[(dval, 'val')], early_stopping_rounds=30,
                          verbose_eval=False)

        risk_scores = model.predict(dval)

        try:
            c_val = concordance_index_ipcw(y_tr_struct, y_va_struct, risk_scores)[0]
            fold_scores.append(c_val)
        except:
            from sksurv.metrics import concordance_index_censored
            c_val = concordance_index_censored(y_va_struct['event'], y_va_struct['time'], risk_scores)[0]
            fold_scores.append(c_val)

        # Clean Memory
        del model, dtrain, dval, risk_scores
        gc.collect()

    # Average & Store
    avg_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)
    results.append({**params, 'Unos_C_Index': avg_score, 'Std_Dev': std_score})

    if (i+1) % 5 == 0:
        elapsed_min = (time.time() - total_start_time) / 60
        best_so_far = max([r['Unos_C_Index'] for r in results])
        nb_print(f"  [{i+1}/{N_ITER}] Best: {best_so_far:.4f} | Current: {avg_score:.4f} | Elapsed: {elapsed_min:.2f} min")

# --- 5. FINALIZE & EXPORT ---
total_duration_min = (time.time() - total_start_time) / 60
nb_print(f"\n🏁 Total Execution Time: {total_duration_min:.2f} minutes")

df_results = pd.DataFrame(results).sort_values(by='Unos_C_Index', ascending=False)
best_config = df_results.iloc[0].to_dict()

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
filename = f"_out/XGB_Readmission_Robust_Tuning_5Fold_{timestamp_str}.csv"

# Ensure directory exists before saving
os.makedirs("_out", exist_ok=True)
df_results.to_csv(filename, index=False)

nb_print(f"\n🏆 Tuning Complete!")
nb_print(f"  Best C-Index: {best_config['Unos_C_Index']:.4f}")
nb_print(f"Saved to: {filename}")

In [24]:
nb_print(best_config)

In [25]:
import pandas as pd
from IPython.display import HTML, display

# Reset options so Pandas doesn't force everything
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Convert DataFrame to HTML and wrap in a scrollable div
html_table = df_results.to_html()
scroll_box = f"""
<div style="max-height:500px; max-width:1000px; overflow-y:auto; overflow-x:auto; border:1px solid #ccc;">
{html_table}
</div>
"""
display(HTML(scroll_box))

,subsample,reg_lambda,reg_alpha,min_child_weight,max_depth,learning_rate,gamma,colsample_bytree,objective,eval_metric,tree_method,seed,nthread,device,verbosity,Unos_C_Index,Std_Dev
14,0.7,1.0,0.1,20,5,0.020,1.0,0.7,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619693,0.007652
26,0.7,5.0,0.0,5,6,0.010,0.0,0.5,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619473,0.007498
21,0.9,10.0,10.0,5,3,0.050,0.0,0.8,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619433,0.007325
79,0.7,10.0,0.0,5,6,0.005,1.0,0.5,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619339,0.006316
27,0.6,20.0,0.0,1,4,0.020,0.1,0.7,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619238,0.008412
72,0.7,10.0,0.0,5,4,0.020,0.1,0.8,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619194,0.007856
91,0.7,1.0,1.0,5,4,0.010,0.0,0.5,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619180,0.006784
31,0.7,20.0,0.0,10,8,0.005,0.5,0.5,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619134,0.006295
90,0.8,5.0,0.1,5,3,0.050,0.1,0.6,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619103,0.007640
2,0.7,5.0,5.0,20,5,0.010,0.5,0.6,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619059,0.007763


In [26]:
#@title 🏆 Optimal XGBoost Configuration (Readmission – Reviewer-Proof Version)
import pandas as pd
from IPython.display import display

data = [
    {"Category": "Performance", "Parameter": "Uno's C-Index (IPCW)", "Value": "0.6197",
     "Description": "Moderate discriminative ability for readmission prediction under right-censoring (5-fold stratified CV)."},

    {"Category": "Stability", "Parameter": "Standard Deviation (CV)", "Value": "±0.0077",
     "Description": "Low cross-validation variability, indicating consistent performance across folds."},

    {"Category": "Tree Structure", "Parameter": "max_depth", "Value": 5,
     "Description": "Moderate tree depth, allowing nonlinear interactions while controlling model complexity."},

    {"Category": "Imbalance Handling", "Parameter": "min_child_weight", "Value": 20,
     "Description": "Restricts small leaf splits, reducing sensitivity to sparse event patterns."},

    {"Category": "Boosting", "Parameter": "learning_rate", "Value": 0.02,
     "Description": "Conservative shrinkage parameter supporting gradual optimization."},

    {"Category": "Tree Structure", "Parameter": "gamma", "Value": 1.0,
     "Description": "Minimum loss reduction required for splits, limiting weak partitioning."},

    {"Category": "Regularization", "Parameter": "reg_alpha (L1)", "Value": 0.1,
     "Description": "Mild L1 regularization to constrain leaf weights and reduce variance."},

    {"Category": "Regularization", "Parameter": "reg_lambda (L2)", "Value": 1.0,
     "Description": "L2 regularization applied to stabilize model estimates."},

    {"Category": "Stochasticity", "Parameter": "subsample", "Value": 0.7,
     "Description": "Row subsampling per tree to reduce variance and improve robustness."},

    {"Category": "Stochasticity", "Parameter": "colsample_bytree", "Value": 0.7,
     "Description": "Feature subsampling per tree to promote diversity among trees."},

    {"Category": "Model Specification", "Parameter": "objective", "Value": "survival:cox",
     "Description": "Cox proportional hazards objective for right-censored time-to-event data."},

    {"Category": "Computation", "Parameter": "tree_method", "Value": "hist",
     "Description": "Histogram-based tree construction for computational efficiency in large datasets."},

    {"Category": "Computation", "Parameter": "CPU threads", "Value": 30,
     "Description": "Parallelized execution using available CPU cores (system reserve applied)."}
]

pd.set_option('display.max_colwidth', None)
df_optimal_config = pd.DataFrame(data)
display(df_optimal_config)

,Category,Parameter,Value,Description
0,Performance,Uno's C-Index (IPCW),0.6197,Moderate discriminative ability for readmission prediction under right-censoring (5-fold stratified CV).
1,Stability,Standard Deviation (CV),±0.0077,"Low cross-validation variability, indicating consistent performance across folds."
2,Tree Structure,max_depth,5,"Moderate tree depth, allowing nonlinear interactions while controlling model complexity."
3,Imbalance Handling,min_child_weight,20,"Restricts small leaf splits, reducing sensitivity to sparse event patterns."
4,Boosting,learning_rate,0.02,Conservative shrinkage parameter supporting gradual optimization.
5,Tree Structure,gamma,1.0,"Minimum loss reduction required for splits, limiting weak partitioning."
6,Regularization,reg_alpha (L1),0.1,Mild L1 regularization to constrain leaf weights and reduce variance.
7,Regularization,reg_lambda (L2),1.0,L2 regularization applied to stabilize model estimates.
8,Stochasticity,subsample,0.7,Row subsampling per tree to reduce variance and improve robustness.
9,Stochasticity,colsample_bytree,0.7,Feature subsampling per tree to promote diversity among trees.


## Optuna

To evaluate calibration in the presence of competing risks, we estimated the absolute probability of readmission using a pseudo-conditional Cumulative Incidence Function (CIF). Because standard Aalen-Johansen estimators are computationally intractable within iterative gradient boosting loops, the CIF was approximated by integrating the XGBoost-derived covariate-adjusted cause-specific hazard for readmission with the marginal Kaplan-Meier estimate of overall event-free survival. This approach ensures competing mortality is mathematically accounted for without strictly requiring the simultaneous estimation of all individual-level competing hazards

🔟 Take-Home Messages

- Tunes XGBoost Cox model for readmission.
- Accounts for competing risk of death.
- Uses 5-fold stratified cross-validation.
- Optimizes discrimination and calibration jointly.
- Computes multi-horizon Uno’s C-index.
- Approximates CIF using Aalen–Johansen logic.
- Uses Brier score based on cumulative incidence.
- Applies Optuna multi-objective Pareto optimization.
- Runs 100 parallel trials (pure search).
- Saves full Optuna study for reproducibility.

🧩 Assumptions (Take-Home Format)

- Cause-specific Cox hazard approximates readmission risk.
- Aalen–Johansen approximation is sufficiently accurate.
- IPCW assumptions for censoring hold.
- Plan-type stratification controls major heterogeneity.
- First imputed dataset reflects full data structure.


In [27]:
# @title Optuna Multi-Objective: C-Index vs IBS (Aalen-Johansen for Competing Risks)
import optuna
import numpy as np
import pandas as pd
import xgboost as xgb
import gc
import os
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_ipcw, brier_score
import joblib
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

nb_print("Preparing data for Competing Risks Multi-Objective Tuning (Pure Search)...")

# --- CPU CONFIGURATION ---
N_CORES = max(1, os.cpu_count() - 2)
nb_print(f"Parallel Execution Configured: Using {N_CORES} CPU cores for Optuna Trials.")

# --- 1. SETUP AND DATA ---
try:
    if 'imputations_list_jan26' in locals():
        df_tune = imputations_list_jan26[0].copy()
    elif 'imputations_list' in locals():
        df_tune = imputations_list[0].copy()
    else:
        df_tune = X_train.copy()
        
    # USING THE CORRECTED READMISSION LIST (Time from Discharge)
    y_readm_struct = y_surv_readm_list[0] 
    y_death_struct = y_surv_death_list[0] 
except Exception as e:
    raise ValueError(f"Data Error: {e}. Please ensure both readmission and death structures are loaded.")

def get_stratification_labels(df):
    labels = np.zeros(len(df), dtype=int)
    if 'plan_type_corr_pg_pr' in df.columns: labels[df['plan_type_corr_pg_pr'] == 1] = 1
    if 'plan_type_corr_m_pr' in df.columns: labels[df['plan_type_corr_m_pr'] == 1] = 2
    if 'plan_type_corr_pg_pai' in df.columns: labels[df['plan_type_corr_pg_pai'] == 1] = 3
    if 'plan_type_corr_m_pai' in df.columns: labels[df['plan_type_corr_m_pai'] == 1] = 4
    return labels

strat_labels = get_stratification_labels(df_tune)
y_xgb_label = np.where(y_readm_struct['event'], y_readm_struct['time'], -y_readm_struct['time'])

# Clinical horizons (months)
EVAL_HORIZONS = [3, 6, 12, 36, 60]

# --- 2. FAST AALEN-JOHANSEN APPROXIMATION ---
def predict_cif_aalen_johansen_approx(y_tr_readm, y_tr_death, risk_tr, risk_va, eval_times):
    """
    Approximates the Cumulative Incidence Function (CIF) using marginal overall survival 
    and patient-specific XGBoost cause-specific hazards.
    """
    if np.any(risk_tr <= 0):
        risk_tr = np.exp(risk_tr)
        risk_va = np.exp(risk_va)

    time_train = y_tr_readm['time']
    # Any event (Readmission OR Death) removes patient from risk pool
    event_any = y_tr_readm['event'] | y_tr_death['event']
    
    order = np.argsort(time_train)
    t_ord = time_train[order]
    e_any_ord = event_any[order]
    e_readm_ord = y_tr_readm['event'][order]
    risk_tr_ord = risk_tr[order]
    
    unique_times = np.unique(t_ord[e_any_ord])
    
    S_all = np.ones(len(unique_times) + 1) # S(t-) overall survival
    baseline_hazard_readm = np.zeros(len(unique_times))
    
    current_S = 1.0
    for i, t in enumerate(unique_times):
        at_risk_mask = t_ord >= t
        n_at_risk_t = np.sum(at_risk_mask)
        
        events_any_t = np.sum((t_ord == t) & e_any_ord)
        events_readm_t = np.sum((t_ord == t) & e_readm_ord)
        
        if n_at_risk_t > 0:
            S_all[i+1] = current_S * (1.0 - events_any_t / n_at_risk_t)
            current_S = S_all[i+1]
            baseline_hazard_readm[i] = events_readm_t / np.sum(risk_tr_ord[at_risk_mask])
            
    cif_va = np.zeros((len(risk_va), len(eval_times)))
    
    for j, eval_t in enumerate(eval_times):
        valid_idx = np.where(unique_times <= eval_t)[0]
        if len(valid_idx) > 0:
            S_all_t_minus = S_all[valid_idx] 
            dH_readm = baseline_hazard_readm[valid_idx]
            base_cif_increment = S_all_t_minus * dH_readm
            cif_va[:, j] = risk_va * np.sum(base_cif_increment)
            
    return 1.0 - cif_va

# --- 3. OPTUNA OBJECTIVE FUNCTION ---
def objective(trial):
    params = {
        'objective': 'survival:cox',
        'eval_metric': 'cox-nloglik',
        'tree_method': 'hist',
        'nthread': 1,               # 🚨 CRITICAL: 1 thread here because Optuna handles parallelization
        'verbosity': 0,
        'seed': 2125,
        # Pure unbiased search space for complex socio-behavioral endpoint
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 25),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 15.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.0, 2.0)
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2125)
    
    fold_c_indices = []
    fold_ib_scores = []
    fold_global_c_indices = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df_tune, strat_labels)):
        X_tr, X_va = df_tune.iloc[train_idx], df_tune.iloc[val_idx]
        y_tr_readm_xgb, y_va_readm_xgb = y_xgb_label[train_idx], y_xgb_label[val_idx]
        
        y_tr_readm_struct, y_va_readm_struct = y_readm_struct[train_idx], y_readm_struct[val_idx]
        y_tr_death_struct = y_death_struct[train_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr_readm_xgb)
        dval = xgb.DMatrix(X_va, label=y_va_readm_xgb)

        model = xgb.train(
            params, dtrain, 
            num_boost_round=1500,
            evals=[(dval, 'val')], 
            early_stopping_rounds=30, 
            verbose_eval=False
        )

        risk_tr = model.predict(dtrain)
        risk_va = model.predict(dval)
        
        # 1. Multi-Horizon C-Index (Cause-Specific)
        h_c_indices = []
        for tau_val in EVAL_HORIZONS:
            try:
                c_val = concordance_index_ipcw(y_tr_readm_struct, y_va_readm_struct, risk_va, tau=tau_val)[0]
                h_c_indices.append(c_val)
            except:
                pass 
        avg_c_index = np.mean(h_c_indices) if len(h_c_indices) > 0 else 0.5
        
        # 2. Global C-Index Tracker
        try:
            global_c = concordance_index_ipcw(y_tr_readm_struct, y_va_readm_struct, risk_va)[0]
        except:
            global_c = 0.5
        fold_global_c_indices.append(global_c)

        # 3. Brier Score using Aalen-Johansen
        try:
            surv_probs_va = predict_cif_aalen_johansen_approx(
                y_tr_readm_struct, y_tr_death_struct, risk_tr, risk_va, EVAL_HORIZONS
            )
            _, brier_scores_at_tau = brier_score(y_tr_readm_struct, y_va_readm_struct, surv_probs_va, EVAL_HORIZONS)
            avg_ibs = np.mean(brier_scores_at_tau)
        except:
            avg_ibs = 0.25 

        fold_c_indices.append(avg_c_index)
        fold_ib_scores.append(avg_ibs)
            
        del model, dtrain, dval, risk_tr, risk_va
        gc.collect()

        # Pruning mechanism
        current_mean_c = np.mean(fold_c_indices)
        if fold_idx >= 1 and current_mean_c < 0.55:
            raise optuna.TrialPruned()

    trial.set_user_attr("Global_C_Index", np.mean(fold_global_c_indices))
    return np.mean(fold_c_indices), np.mean(fold_ib_scores)

# --- 4. INITIALIZATION AND EXECUTION ---
study = optuna.create_study(
    directions=['maximize', 'minimize'], 
    study_name="XGB_Readm_Pareto_Fresh"
)

# NO WARM START - 100% Pure Objective Search
nb_print("Starting Pure Multi-Objective Optimization for Readmission (Aalen-Johansen Brier Score)")

# 🚨 n_jobs=N_CORES tells Optuna to run multiple configurations simultaneously
study.optimize(objective, n_trials=100, n_jobs=N_CORES, show_progress_bar=True)

# --- 5. EXTRACTION OF OPTIMAL MODELS ---
nb_print("\nOptimal Models found (Pareto Front):")
best_trials = study.best_trials
for t in best_trials:
    global_c_val = t.user_attrs.get("Global_C_Index", "N/A")
    nb_print(f"Trial {t.number} -> Multi-Horizon C: {t.values[0]:.4f} | IBS: {t.values[1]:.4f} | Global C: {global_c_val:.4f}")

# --- 6. SAVE THE ENTIRE STUDY TO A .PKL FILE ---
os.makedirs("_input", exist_ok=True)
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
study_filename = f"_input/XGB_Readm_Optuna_Study_Fresh_{timestamp_str}.pkl"

joblib.dump(study, study_filename)
nb_print(f"\nStudy object successfully saved to: {study_filename}")

~47 minutes

In [29]:
# @title Final Model Selection for Readmission (Euclidean Distance to Ideal Point)
import os
import glob
import re
import joblib
import pandas as pd
import numpy as np
from datetime import datetime

nb_print("Analyzing the Pareto Front for Readmission...")

# --- Load or reuse study object (minimal changes) ---
# If an in-memory study with the specific study_name exists, use it.
# Otherwise, find the most recent saved study file matching the pattern and load it.
expected_study_name = "XGB_Readm_Pareto_Fresh"
if 'study' in globals() and getattr(study, 'study_name', None) == expected_study_name:
    print(f"Using in-memory study with study_name='{expected_study_name}'.")
else:
    os.makedirs("_input", exist_ok=True)
    pattern = "_input/XGB_Readm_Optuna_Study_Fresh_*.pkl"
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(f"No saved study files found matching pattern: {pattern}")
    timestamped_files = []
    for f in files:
        m = re.search(r"XGB_Readm_Optuna_Study_Fresh_(\d{8}_\d{4})\.pkl$", f)
        if m:
            ts = m.group(1)
            try:
                dt = datetime.strptime(ts, "%Y%m%d_%H%M")
                timestamped_files.append((dt, f))
            except ValueError:
                continue
    if not timestamped_files:
        raise FileNotFoundError("No study files with a valid timestamp found in filenames.")
    latest_file = max(timestamped_files, key=lambda x: x[0])[1]
    study = joblib.load(latest_file)
    nb_print(f"Loaded study object from most recent file: {latest_file}")

# 1. Extract all optimal models (Pareto Front)
pareto_trials = study.best_trials

# 2. Convert the optimal trials into a DataFrame
pareto_data = []
for t in pareto_trials:
    row = {
        "trial_id": t.number,
        "Multi_Horizon_C_Index": t.values[0],
        "Aalen_Johansen_Brier_Score": t.values[1],
        "Global_C_Index": t.user_attrs.get("Global_C_Index", np.nan)
    }
    row.update(t.params)
    pareto_data.append(row)

df_pareto = pd.DataFrame(pareto_data)

# 3. Calculate the Distance to the Ideal Point
# Ideal point: C-Index = 1.0, Brier Score = 0.0
df_pareto["Distance_to_Ideal"] = np.sqrt(
    (1.0 - df_pareto["Multi_Horizon_C_Index"])**2 + (df_pareto["Aalen_Johansen_Brier_Score"])**2
)

# 4. Sort to find the absolute winner (the knee point)
df_pareto = df_pareto.sort_values("Distance_to_Ideal", ascending=True).reset_index(drop=True)

# --- RESULTS ---
nb_print(f"\nFound {len(df_pareto)} non-dominated models in the Pareto Front.")

nb_print("\n🏆 ABSOLUTE WINNER (Optimal Trade-off):")
winner = df_pareto.iloc[0]
nb_print(f"  Trial ID              : {winner['trial_id']}")
nb_print(f"  Multi-Horizon C-Index : {winner['Multi_Horizon_C_Index']:.4f}")
nb_print(f"  Aalen-Johansen Brier  : {winner['Aalen_Johansen_Brier_Score']:.4f}")
nb_print(f"  Global C-Index        : {winner['Global_C_Index']:.4f}")
nb_print(f"  Distance to Ideal     : {winner['Distance_to_Ideal']:.4f}")

nb_print("\n⚙️ Winner Hyperparameters:")
exclude_keys = ["trial_id", "Multi_Horizon_C_Index", "Aalen_Johansen_Brier_Score", "Global_C_Index", "Distance_to_Ideal"]
params_winner = {k: v for k, v in winner.items() if k not in exclude_keys}
for k, v in params_winner.items():
    nb_print(f"  {k}: {v}")

# Export
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
os.makedirs("_out", exist_ok=True)
filename_pareto = f"_out/Readmission_Pareto_Front_{timestamp_str}.csv"
df_pareto.to_csv(filename_pareto, index=False)
nb_print(f"\n💾 Full Pareto Front saved to: {filename_pareto}")

In [30]:
import pandas as pd
from IPython.display import HTML, display

# Reset options so Pandas doesn't force everything
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Convert DataFrame to HTML and wrap in a scrollable div
html_table_opt1 = df_results.to_html()
scroll_box_opt1 = f"""
<div style="max-height:500px; max-width:1000px; overflow-y:auto; overflow-x:auto; border:1px solid #ccc;">
{html_table_opt1}
</div>
"""
display(HTML(scroll_box_opt1))

,subsample,reg_lambda,reg_alpha,min_child_weight,max_depth,learning_rate,gamma,colsample_bytree,objective,eval_metric,tree_method,seed,nthread,device,verbosity,Unos_C_Index,Std_Dev
14,0.7,1.0,0.1,20,5,0.020,1.0,0.7,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619693,0.007652
26,0.7,5.0,0.0,5,6,0.010,0.0,0.5,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619473,0.007498
21,0.9,10.0,10.0,5,3,0.050,0.0,0.8,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619433,0.007325
79,0.7,10.0,0.0,5,6,0.005,1.0,0.5,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619339,0.006316
27,0.6,20.0,0.0,1,4,0.020,0.1,0.7,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619238,0.008412
72,0.7,10.0,0.0,5,4,0.020,0.1,0.8,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619194,0.007856
91,0.7,1.0,1.0,5,4,0.010,0.0,0.5,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619180,0.006784
31,0.7,20.0,0.0,10,8,0.005,0.5,0.5,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619134,0.006295
90,0.8,5.0,0.1,5,3,0.050,0.1,0.6,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619103,0.007640
2,0.7,5.0,5.0,20,5,0.010,0.5,0.6,survival:cox,cox-nloglik,hist,2125,30,cpu,0,0.619059,0.007763


### Optuna (2)

1. Models readmission hazard via XGBoost cause-specific Cox.

2. Death treated as competing risk, not simple censoring.

3. Uses **dual stratification (plan + event) in CV**.

4. Optimizes discrimination and calibration jointly.

5. Multi-horizon IPCW C-index guides discrimination.

6. Aalen–Johansen approximation used for CIF.

7. Brier score evaluates absolute risk calibration.

8. Optuna NSGA-II finds Pareto-optimal models.

9. Early pruning avoids wasting trials.

10. Balances flexibility and regularization via Bayesian search.

**Key assumptions**

1. Proportional hazards for readmission risk.

2. Independent censoring conditional on covariates.

3. CIF approximation is numerically stable.

4. Dual stratification preserves fold representativeness.

5. Imputed datasets adequately reflect missing data uncertainty.

In [32]:
# @title Optuna Multi-Objective: C-Index vs IBS (Aalen-Johansen for Competing Risks)
import optuna
import numpy as np
import pandas as pd
import xgboost as xgb
import gc
import os
import joblib
from datetime import datetime
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_ipcw, brier_score
import warnings

warnings.filterwarnings("ignore")

nb_print("Preparing data for Competing Risks Multi-Objective Tuning (Dual Stratified)...")

# --- CPU CONFIGURATION ---
N_CORES = max(1, os.cpu_count() - 2)
nb_print(f"Parallel Execution Configured: Using {N_CORES} CPU cores for concurrent Optuna Trials.")

# --- 1. SETUP AND DATA ---
try:
    if 'imputations_list_jan26' in locals():
        df_tune = imputations_list_jan26[0].copy()
    elif 'imputations_list' in locals():
        df_tune = imputations_list[0].copy()
    else:
        df_tune = X_train.copy()
        
    # As requested: using uncorrected readmission vector for competing risks alignment
    y_readm_struct = y_surv_readm_list[0] 
    y_death_struct = y_surv_death_list[0] 
except Exception as e:
    raise ValueError(f"Data Error: {e}. Please ensure both readmission and death structures are loaded.")

# DUAL STRATIFICATION (Plan + Event)
def get_dual_stratification_labels(df, y_struct):
    labels = np.zeros(len(df), dtype=int)
    if 'plan_type_corr_pg_pr' in df.columns: labels[df['plan_type_corr_pg_pr'] == 1] = 1
    if 'plan_type_corr_m_pr' in df.columns: labels[df['plan_type_corr_m_pr'] == 1] = 2
    if 'plan_type_corr_pg_pai' in df.columns: labels[df['plan_type_corr_pg_pai'] == 1] = 3
    if 'plan_type_corr_m_pai' in df.columns: labels[df['plan_type_corr_m_pai'] == 1] = 4
    
    event_status = y_struct['event'].astype(int)
    return (labels * 10) + event_status

strat_labels_dual = get_dual_stratification_labels(df_tune, y_readm_struct)
y_xgb_label = np.where(y_readm_struct['event'], y_readm_struct['time'], -y_readm_struct['time'])

# Clinical horizons (months)
EVAL_HORIZONS = [3, 6, 12, 36, 60]

# --- 2. FAST AALEN-JOHANSEN APPROXIMATION ---
def predict_cif_aalen_johansen_approx(y_tr_readm, y_tr_death, risk_tr, risk_va, eval_times):
    if np.any(risk_tr <= 0):
        risk_tr = np.exp(risk_tr)
        risk_va = np.exp(risk_va)

    time_train = y_tr_readm['time']
    event_any = y_tr_readm['event'] | y_tr_death['event']
    
    order = np.argsort(time_train)
    t_ord = time_train[order]
    e_any_ord = event_any[order]
    e_readm_ord = y_tr_readm['event'][order]
    risk_tr_ord = risk_tr[order]
    
    unique_times = np.unique(t_ord[e_any_ord])
    S_all = np.ones(len(unique_times) + 1) 
    baseline_hazard_readm = np.zeros(len(unique_times))
    
    current_S = 1.0
    for i, t in enumerate(unique_times):
        at_risk_mask = t_ord >= t
        n_at_risk_t = np.sum(at_risk_mask)
        events_any_t = np.sum((t_ord == t) & e_any_ord)
        events_readm_t = np.sum((t_ord == t) & e_readm_ord)
        
        if n_at_risk_t > 0:
            S_all[i+1] = current_S * (1.0 - events_any_t / n_at_risk_t)
            current_S = S_all[i+1]
            baseline_hazard_readm[i] = events_readm_t / np.sum(risk_tr_ord[at_risk_mask])
            
    cif_va = np.zeros((len(risk_va), len(eval_times)))
    
    for j, eval_t in enumerate(eval_times):
        valid_idx = np.where(unique_times <= eval_t)[0]
        if len(valid_idx) > 0:
            S_all_t_minus = S_all[valid_idx] 
            dH_readm = baseline_hazard_readm[valid_idx]
            base_cif_increment = S_all_t_minus * dH_readm
            cif_va[:, j] = risk_va * np.sum(base_cif_increment)
            
    return 1.0 - cif_va

# --- 3. OPTUNA OBJECTIVE FUNCTION ---
def objective(trial):
    params = {
        'objective': 'survival:cox',
        'eval_metric': 'cox-nloglik',
        'tree_method': 'hist',
        'nthread': 1,               # STRICTLY 1 THREAD PER MODEL to allow n_jobs at study level
        'verbosity': 0,
        'seed': 2125,
        
        # Search space centered around optimal ranges
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 30),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 15.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.0, 2.0)
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2125)
    
    fold_c_indices = []
    fold_ib_scores = []
    fold_global_c_indices = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df_tune, strat_labels_dual)):
        X_tr, X_va = df_tune.iloc[train_idx], df_tune.iloc[val_idx]
        y_tr_readm_xgb, y_va_readm_xgb = y_xgb_label[train_idx], y_xgb_label[val_idx]
        
        y_tr_readm_struct, y_va_readm_struct = y_readm_struct[train_idx], y_readm_struct[val_idx]
        y_tr_death_struct = y_death_struct[train_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr_readm_xgb)
        dval = xgb.DMatrix(X_va, label=y_va_readm_xgb)

        model = xgb.train(
            params, dtrain, 
            num_boost_round=1500,
            evals=[(dval, 'val')], 
            early_stopping_rounds=30, 
            verbose_eval=False
        )

        risk_tr = model.predict(dtrain)
        risk_va = model.predict(dval)
        
        # 1. Multi-Horizon C-Index
        h_c_indices = []
        for tau_val in EVAL_HORIZONS:
            try:
                c_val = concordance_index_ipcw(y_tr_readm_struct, y_va_readm_struct, risk_va, tau=tau_val)[0]
                h_c_indices.append(c_val)
            except:
                pass 
        avg_c_index = np.mean(h_c_indices) if len(h_c_indices) > 0 else 0.5
        
        # 2. Global C-Index Tracker
        try:
            global_c = concordance_index_ipcw(y_tr_readm_struct, y_va_readm_struct, risk_va)[0]
        except:
            global_c = 0.5
        fold_global_c_indices.append(global_c)

        # 3. Brier Score using Aalen-Johansen
        try:
            surv_probs_va = predict_cif_aalen_johansen_approx(
                y_tr_readm_struct, y_tr_death_struct, risk_tr, risk_va, EVAL_HORIZONS
            )
            _, brier_scores_at_tau = brier_score(y_tr_readm_struct, y_va_readm_struct, surv_probs_va, EVAL_HORIZONS)
            avg_ibs = np.mean(brier_scores_at_tau)
        except:
            avg_ibs = 0.25 

        fold_c_indices.append(avg_c_index)
        fold_ib_scores.append(avg_ibs)
            
        del model, dtrain, dval, risk_tr, risk_va
        gc.collect()

        # Pruning mechanism
        current_mean_c = np.mean(fold_c_indices)
        if fold_idx >= 1 and current_mean_c < 0.55:
            raise optuna.TrialPruned()

    trial.set_user_attr("Global_C_Index", np.mean(fold_global_c_indices))
    return np.mean(fold_c_indices), np.mean(fold_ib_scores)

# --- 4. INITIALIZATION AND EXECUTION ---
study = optuna.create_study(
    directions=['maximize', 'minimize'], 
    study_name="XGB_Readm_Pareto_AJ_DualStrat"
)

# Prime the search with the winner from the previous robust tuning step
best_prior_config = {
    'subsample': 0.7, 
    'reg_lambda': 1.0, 
    'reg_alpha': 0.1, 
    'min_child_weight': 20, 
    'max_depth': 5, 
    'learning_rate': 0.02, 
    'gamma': 1.0, 
    'colsample_bytree': 0.7
}
study.enqueue_trial(best_prior_config)

nb_print("Starting Multi-Objective Optimization for Readmission (Aalen-Johansen & Dual Stratification)")
# Adjusted n_trials to 50 as requested
study.optimize(objective, n_trials=50, n_jobs=N_CORES, show_progress_bar=True)

# --- 5. EXTRACTION AND SAVING ---
nb_print("\nOptimal Models found (Pareto Front):")
best_trials = study.best_trials
for t in best_trials:
    global_c_val = t.user_attrs.get("Global_C_Index", "N/A")
    nb_print(f"Trial {t.number} -> Multi-Horizon C: {t.values[0]:.4f} | IBS: {t.values[1]:.4f} | Global C: {global_c_val:.4f}")

os.makedirs("_input", exist_ok=True)
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
study_filename = f"_input/XGB_Readm_Optuna2_Study_AJ_{timestamp_str}.pkl"

joblib.dump(study, study_filename)
nb_print(f"\nStudy object successfully saved to: {study_filename}")

~33 minutes

In [35]:
# @title Final Model Selection for Readmission (Euclidean Distance to Ideal Point)
import os
import glob
import re
import joblib
import pandas as pd
import numpy as np
from datetime import datetime

nb_print("Analyzing the Dual-Stratified Pareto Front for Readmission...")

# --- Load or reuse study object (minimal changes) ---
expected_study_name = "XGB_Readm_Pareto_AJ_DualStrat"
if 'study' in globals() and getattr(study, 'study_name', None) == expected_study_name:
    nb_print(f"Using in-memory study with study_name='{expected_study_name}'.")
else:
    os.makedirs("_input", exist_ok=True)
    pattern = "_input/XGB_Readm_Optuna2_Study_AJ_*.pkl"
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(f"No saved study files found matching pattern: {pattern}")
    timestamped_files = []
    for f in files:
        m = re.search(r"XGB_Readm_Optuna2_Study_AJ_(\d{8}_\d{4})\.pkl$", f)
        if m:
            ts = m.group(1)
            try:
                dt = datetime.strptime(ts, "%Y%m%d_%H%M")
                timestamped_files.append((dt, f))
            except ValueError:
                continue
    if not timestamped_files:
        raise FileNotFoundError("No study files with a valid timestamp found in filenames.")
    latest_file = max(timestamped_files, key=lambda x: x[0])[1]
    study = joblib.load(latest_file)
    nb_print(f"Loaded study object from most recent file: {latest_file}")

# 1. Extract all optimal models (Pareto Front)
pareto_trials = study.best_trials

# 2. Convert the optimal trials into a DataFrame
pareto_data = []
for t in pareto_trials:
    row = {
        "trial_id": t.number,
        "Multi_Horizon_C_Index": t.values[0],
        "Aalen_Johansen_Brier_Score": t.values[1],
        "Global_C_Index": t.user_attrs.get("Global_C_Index", np.nan)
    }
    row.update(t.params)
    pareto_data.append(row)

df_pareto = pd.DataFrame(pareto_data)

# 3. Calculate the Distance to the Ideal Point
# Ideal point: C-Index = 1.0, Brier Score = 0.0
df_pareto["Distance_to_Ideal"] = np.sqrt(
    (1.0 - df_pareto["Multi_Horizon_C_Index"])**2 + (df_pareto["Aalen_Johansen_Brier_Score"])**2
)

# 4. Sort to find the absolute winner (the knee point)
df_pareto = df_pareto.sort_values("Distance_to_Ideal", ascending=True).reset_index(drop=True)

# --- RESULTS ---
nb_print(f"\nFound {len(df_pareto)} non-dominated models in the Pareto Front.")

nb_print("\nABSOLUTE WINNER (Optimal Trade-off):")
winner = df_pareto.iloc[0]
nb_print(f"  Trial ID              : {winner['trial_id']}")
nb_print(f"  Multi-Horizon C-Index : {winner['Multi_Horizon_C_Index']:.4f}")
nb_print(f"  Aalen-Johansen Brier  : {winner['Aalen_Johansen_Brier_Score']:.4f}")
nb_print(f"  Global C-Index        : {winner['Global_C_Index']:.4f}")
nb_print(f"  Distance to Ideal     : {winner['Distance_to_Ideal']:.4f}")

print("\nWinner Hyperparameters:")
exclude_keys = ["trial_id", "Multi_Horizon_C_Index", "Aalen_Johansen_Brier_Score", "Global_C_Index", "Distance_to_Ideal"]
params_winner = {k: v for k, v in winner.items() if k not in exclude_keys}
for k, v in params_winner.items():
    nb_print(f"  {k}: {v}")

# Export 
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
os.makedirs("_out", exist_ok=True)
filename_pareto = f"_out/Readmission_Pareto2_Front_DualStrat_{timestamp_str}.csv"
df_pareto.to_csv(filename_pareto, index=False)
nb_print(f"\nFull Pareto Front saved to: {filename_pareto}")


In [36]:

import pandas as pd
from IPython.display import HTML, display

# Reset options so Pandas doesn't force everything
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Convert DataFrame to HTML and wrap in a scrollable div
html_table_opt2 = df_pareto.to_html()
scroll_box_opt2 = f"""
<div style="max-height:500px; max-width:1000px; overflow-y:auto; overflow-x:auto; border:1px solid #ccc;">
{html_table_opt2}
</div>
"""
display(HTML(scroll_box_opt2))

,trial_id,Multi_Horizon_C_Index,Aalen_Johansen_Brier_Score,Global_C_Index,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,gamma,Distance_to_Ideal
0,39,0.649499,0.110482,0.618921,0.009309,9,20,0.834674,0.706241,1.324474,5.310034,0.229114,0.367502
1,9,0.649083,0.109875,0.619820,0.002874,9,12,0.740598,0.660596,0.224741,4.115217,0.007231,0.367716
2,19,0.648549,0.109643,0.619945,0.003668,7,20,0.787309,0.500536,0.011614,1.603510,1.430836,0.368157
3,20,0.646419,0.109358,0.617690,0.001688,9,12,0.649461,0.717653,3.449988,1.002501,1.887752,0.370106
4,2,0.645448,0.109214,0.616207,0.001070,8,13,0.514954,0.632174,0.250694,1.283056,1.591510,0.370992


### 📊 IPython: Summary of Pareto Findings

In [38]:
from IPython.display import display, HTML

html_content = """
<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6; color: #333; max-width: 850px;">

<h2 style="color: #2c3e50; border-bottom: 2px solid #ecf0f1; padding-bottom: 5px;">🧠 Decoding the Readmission Pareto Front</h2>
<p style="font-size: 15px;">
Analysis of the top 5 Pareto-optimal configurations for the readmission outcome under a competing risks framework (Aalen-Johansen). The hyperparameter topography reveals a distinct structural requirement compared to the mortality endpoint.
</p>

<table style="width: 100%; border-collapse: collapse; margin-top: 20px;">
    <tr>
        <td style="width: 50%; vertical-align: top; padding-right: 15px;">
            <h3 style="color: #2980b9;">🌳 1. High Dimensionality (Depth)</h3>
            <p style="font-size: 14px;">
                <b>Max Depth:</b> The Pareto front exclusively selects very deep trees (ranging from <code>7</code> to <code>9</code>). <br>
                <i>Statistical Meaning:</i> Unlike mortality (which resolved at depth 3), readmission prediction relies heavily on high-order interactions. The model requires multiple conditional splits to identify at-risk subgroups.
            </p>
        </td>
        <td style="width: 50%; vertical-align: top; padding-left: 15px; border-left: 1px solid #eee;">
            <h3 style="color: #2980b9;">🐢 2. Ultra-Slow Convergence</h3>
            <p style="font-size: 14px;">
                <b>Learning Rate:</b> Values are strictly below 0.01 (<code>0.001</code> to <code>0.009</code>).<br>
                <i>Statistical Meaning:</i> Because the trees are deep, the risk of overfitting is severe. The algorithm compensates by making the contribution of each individual tree microscopic, forcing a slow, ensemble-driven consensus.
            </p>
        </td>
    </tr>
    <tr>
        <td style="width: 50%; vertical-align: top; padding-right: 15px; padding-top: 15px;">
            <h3 style="color: #2980b9;">🛡️ 3. Strict Nodal Constraints</h3>
            <p style="font-size: 14px;">
                <b>Min Child Weight:</b> Consistently high (<code>12</code> to <code>20</code>).<br>
                <i>Statistical Meaning:</i> Acts as a direct counterbalance to the high tree depth. While the model is allowed to search for complex interactions, it is strictly forbidden from creating terminal nodes (leaves) that represent only a handful of patients, preserving generalizability.
            </p>
        </td>
        <td style="width: 50%; vertical-align: top; padding-left: 15px; padding-top: 15px; border-left: 1px solid #eee;">
            <h3 style="color: #2980b9;">⚖️ 4. Heavy Regularization</h3>
            <p style="font-size: 14px;">
                <b>L1 & L2 Penalties:</b> Substantial L2 (Ridge) penalties (up to <code>5.31</code> in the winning model).<br>
                <i>Statistical Meaning:</i> Further controls the variance introduced by the depth of the trees, shrinking the leaf weights smoothly to prevent extreme hazard predictions.
            </p>
        </td>
    </tr>
</table>

<div style="background-color: #f8f9fa; border-left: 5px solid #2c3e50; padding: 15px; margin-top: 25px; border-radius: 0 5px 5px 0;">
    <h4 style="margin-top: 0; color: #2c3e50; font-size: 16px;">💡 Methodological Takeaway</h4>
    <p style="margin-bottom: 0; font-size: 15px;">
        The algorithm clearly identifies readmission as a highly complex, non-linear socio-behavioral event. To model it, XGBoost demands deep architectural flexibility (depth 9) but requires aggressive regularization (low learning rate, high child weight, strong L2) to prevent the memorization of sample-specific noise. The resulting Multi-Horizon C-Index of ~0.649 reflects a moderate, realistic discriminative capacity for SUD treatment readmission.
    </p>
</div>

</div>
"""

display(HTML(html_content))

"🌳 1. High Dimensionality (Depth) Max Depth: The Pareto front exclusively selects very deep trees (ranging from 7 to 9). Statistical Meaning: Unlike mortality (which resolved at depth 3), readmission prediction relies heavily on high-order interactions. The model requires multiple conditional splits to identify at-risk subgroups.","🐢 2. Ultra-Slow Convergence Learning Rate: Values are strictly below 0.01 (0.001 to 0.009). Statistical Meaning: Because the trees are deep, the risk of overfitting is severe. The algorithm compensates by making the contribution of each individual tree microscopic, forcing a slow, ensemble-driven consensus."
"🛡️ 3. Strict Nodal Constraints Min Child Weight: Consistently high (12 to 20). Statistical Meaning: Acts as a direct counterbalance to the high tree depth. While the model is allowed to search for complex interactions, it is strictly forbidden from creating terminal nodes (leaves) that represent only a handful of patients, preserving generalizability.","⚖️ 4. Heavy Regularization L1 & L2 Penalties: Substantial L2 (Ridge) penalties (up to 5.31 in the winning model). Statistical Meaning: Further controls the variance introduced by the depth of the trees, shrinking the leaf weights smoothly to prevent extreme hazard predictions."


### optimism correction

In [39]:
# @title Harrell's Bootstrap Optimism Correction (Readmission, Parallelized, CPU-2)
import numpy as np
import pandas as pd
import xgboost as xgb
import gc
import os
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sksurv.metrics import concordance_index_ipcw
from joblib import Parallel, delayed
import warnings

warnings.filterwarnings("ignore")

# Fallback for nb_print
if 'nb_print' not in globals():
    def nb_print(*args, **kwargs):
        print(*args, **kwargs)

nb_print("Initializing Parallel Harrell's Bootstrap Optimism Correction for Readmission...")

# --- CPU CONFIGURATION ---
N_CORES = max(1, os.cpu_count() - 2)
nb_print(f"Parallel Execution Configured: Using {N_CORES} CPU cores.")

# --- 1. DATA SETUP (Strict adherence to Competing Risks definitions) ---
try:
    if 'imputations_list_jan26' in locals():
        df_tune = imputations_list_jan26[0].copy()
    elif 'imputations_list' in locals():
        df_tune = imputations_list[0].copy()
    else:
        df_tune = X_train.copy()
        
    # Using the exact array used in the Aalen-Johansen tuning step
    y_tune_struct = y_surv_readm_list[0] 
except Exception as e:
    raise ValueError(f"Data Error: {e}. Please ensure readmission structures are loaded.")

y_xgb_label = np.where(y_tune_struct['event'], y_tune_struct['time'], -y_tune_struct['time'])

# --- 2. SET TRIAL 39 HYPERPARAMETERS (ABSOLUTE WINNER) ---
params_winner = {
    'objective': 'survival:cox',
    'eval_metric': 'cox-nloglik',
    'tree_method': 'hist',
    'device': 'cpu',
    'verbosity': 0,
    'seed': 42,
    
    # Trial 39 parameters from Pareto Front
    'learning_rate': 0.009309,
    'max_depth': 9,
    'min_child_weight': 20,
    'subsample': 0.834674,
    'colsample_bytree': 0.706241,
    'reg_alpha': 1.324474,
    'reg_lambda': 5.310034,
    'gamma': 0.229114
}

B_ITERATIONS = 500 # Robust clinical standard

# --- STRATIFICATION HELPER ---
def get_dual_stratification_labels(df, y_struct):
    labels = np.zeros(len(df), dtype=int)
    if 'plan_type_corr_pg_pr' in df.columns: labels[df['plan_type_corr_pg_pr'] == 1] = 1
    if 'plan_type_corr_m_pr' in df.columns: labels[df['plan_type_corr_m_pr'] == 1] = 2
    if 'plan_type_corr_pg_pai' in df.columns: labels[df['plan_type_corr_pg_pai'] == 1] = 3
    if 'plan_type_corr_m_pai' in df.columns: labels[df['plan_type_corr_m_pai'] == 1] = 4
    event_status = y_struct['event'].astype(int)
    return (labels * 10) + event_status

strat_labels_dual = get_dual_stratification_labels(df_tune, y_tune_struct)

# --- 3. CALCULATE APPARENT PERFORMANCE ON ORIGINAL DATA ---
nb_print("Calculating apparent performance on the original full dataset...")

params_initial = params_winner.copy()
params_initial['nthread'] = N_CORES

X_train_app, X_val_app, y_train_xgb_app, y_val_xgb_app = train_test_split(
    df_tune, y_xgb_label, test_size=0.2, random_state=42, stratify=strat_labels_dual
)

dtrain_app = xgb.DMatrix(X_train_app, label=y_train_xgb_app)
dval_app = xgb.DMatrix(X_val_app, label=y_val_xgb_app)

temp_model = xgb.train(
    params_initial, dtrain_app, 
    num_boost_round=2000, # Increased upper limit due to very low LR
    evals=[(dval_app, 'val')], 
    early_stopping_rounds=30, 
    verbose_eval=False
)
optimal_boost_rounds = temp_model.best_iteration

nb_print(f"Optimal boosting rounds determined: {optimal_boost_rounds}")

# Train the definitive baseline model on 100% of the data
dorig = xgb.DMatrix(df_tune, label=y_xgb_label)
baseline_model = xgb.train(
    params_initial, dorig, 
    num_boost_round=optimal_boost_rounds,
    verbose_eval=False
)

risk_orig = baseline_model.predict(dorig)
try:
    c_apparent_orig = concordance_index_ipcw(y_tune_struct, y_tune_struct, risk_orig)[0]
except:
    from sksurv.metrics import concordance_index_censored
    c_apparent_orig = concordance_index_censored(y_tune_struct['event'], y_tune_struct['time'], risk_orig)[0]

nb_print(f"Baseline Apparent Global C-index: {c_apparent_orig:.4f}")

# --- 4. PARALLEL BOOTSTRAP WORKER FUNCTION ---
def parallel_bootstrap_worker(b, df_original, y_xgb_original, y_struct_orig, params, opt_rounds):
    # CRITICAL: 1 thread per XGBoost model to prevent CPU thrashing
    boot_params = params.copy()
    boot_params['nthread'] = 1 
    
    indices = np.arange(len(df_original))
    boot_indices = resample(indices, replace=True, n_samples=len(indices), random_state=b)
    
    X_boot = df_original.iloc[boot_indices]
    y_xgb_boot = y_xgb_original[boot_indices]
    y_struct_boot = y_struct_orig[boot_indices]
    
    dboot = xgb.DMatrix(X_boot, label=y_xgb_boot)
    dorig_local = xgb.DMatrix(df_original, label=y_xgb_original)
    
    boot_model = xgb.train(
        boot_params, dboot, 
        num_boost_round=opt_rounds,
        verbose_eval=False
    )
    
    # Apparent Boot Performance
    risk_boot = boot_model.predict(dboot)
    try:
        c_boot_app = concordance_index_ipcw(y_struct_boot, y_struct_boot, risk_boot)[0]
    except:
        from sksurv.metrics import concordance_index_censored
        c_boot_app = concordance_index_censored(y_struct_boot['event'], y_struct_boot['time'], risk_boot)[0]
        
    # Test Performance on Original Data
    risk_test_orig = boot_model.predict(dorig_local)
    try:
        c_boot_test = concordance_index_ipcw(y_struct_boot, y_struct_orig, risk_test_orig)[0]
    except:
        from sksurv.metrics import concordance_index_censored
        c_boot_test = concordance_index_censored(y_struct_orig['event'], y_struct_orig['time'], risk_test_orig)[0]
        
    optimism = c_boot_app - c_boot_test
    return optimism

# --- 5. EXECUTE PARALLEL BOOTSTRAP LOOP ---
nb_print(f"\nLaunching {B_ITERATIONS} Parallel Bootstrap Iterations...")

optimism_values = Parallel(n_jobs=N_CORES, verbose=10)(
    delayed(parallel_bootstrap_worker)(
        b, df_tune, y_xgb_label, y_tune_struct, params_winner, optimal_boost_rounds
    ) for b in range(B_ITERATIONS)
)

# --- 6. CALCULATE FINAL CORRECTED METRICS ---
mean_optimism = np.mean(optimism_values)
c_index_corrected = c_apparent_orig - mean_optimism

nb_print("\n--------------------------------------------------")
nb_print("FINAL OPTIMISM-CORRECTED RESULTS (READMISSION)")
nb_print("--------------------------------------------------")
nb_print(f"Apparent C-Index (Original Data) : {c_apparent_orig:.4f}")
nb_print(f"Mean Optimism (from {B_ITERATIONS} boots)   : {mean_optimism:.4f}")
nb_print(f"Optimism-Corrected C-Index       : {c_index_corrected:.4f}")
nb_print("--------------------------------------------------")

# Export
os.makedirs("_out", exist_ok=True)
results_df = pd.DataFrame({
    'Metric': ['Apparent_C_Index', 'Mean_Optimism', 'Corrected_C_Index'],
    'Value': [c_apparent_orig, mean_optimism, c_index_corrected]
})
timestamp_str = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
filename = f"_out/XGB_Readm_Bootstrap_Optimism_Results_{timestamp_str}.csv"
results_df.to_csv(filename, index=False)
nb_print(f"Results saved successfully to {filename}.")

~32 minutes

#### Readmission Model – Internal Validation Summary

- Apparent performance overestimated discrimination by ~0.06 C-index units.
- Bootstrap-corrected C-index stabilized at 0.628 (The optimism-corrected global Uno’s C-index across the full follow-up distribution.)
- Corrected performance aligns closely with cross-validation results.
- Overfitting was moderate but appropriately corrected.
- Readmission prediction shows structurally lower stability than mortality.
- Flexible tree-based models inflate apparent performance without correction.
- Internal validation confirms a realistic discrimination ceiling (~0.63).
- The model demonstrates moderate, clinically plausible discrimination.
- Optimism correction strengthens methodological credibility.
- Results are suitable for transparent reporting under TRIPOD guidelines.

In [43]:
from IPython.display import HTML, display
html_table = results_df.to_html(index=True, escape=False)
scroll_box = f"""
<div style="max-height:600px; max-width:100%; overflow-y:auto; overflow-x:auto; border:1px solid #ddd; padding:6px;">
{html_table}
</div>
"""
display(HTML(scroll_box))


,Metric,Value
0,Apparent_C_Index,0.686409
1,Mean_Optimism,0.058583
2,Corrected_C_Index,0.627825
